# House Price Prediction
This notebook loads the housing dataset configured in `conf/config.yml`, prepares mixed numeric and categorical features, trains a regression model, and evaluates held-out predictions.

Place the housing CSV at `data/raw/housing/housing.csv` and include a `SalePrice` target column before running the notebook.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from ta_lib.custom import SignedLog1pTransformer

NOTEBOOK_DIR = Path.cwd()
with (NOTEBOOK_DIR / 'conf' / 'config.yml').open() as config_file:
    config = yaml.safe_load(config_file)

data_path = NOTEBOOK_DIR / config['dataset_path']
housing = pd.read_csv(data_path)
target_column = config['target_column']
if target_column not in housing.columns:
    raise KeyError(f'Missing target column: {target_column}')

housing.head()

In [ ]:
X = housing.drop(columns=[target_column])
y = housing[target_column]
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('signed_log1p', SignedLog1pTransformer()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config['test_size'], random_state=config['random_state']
)

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=config['random_state'],
        n_jobs=-1,
        **config['model'],
    )),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {
    'mae': mean_absolute_error(y_test, predictions),
    'rmse': mean_squared_error(y_test, predictions) ** 0.5,
    'r2': r2_score(y_test, predictions),
}
pd.Series(metrics, name='value')

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.55)
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(limits, limits, linestyle='--', color='black')
plt.xlabel('Actual sale price')
plt.ylabel('Predicted sale price')
plt.title('House price predictions')
plt.tight_layout()
plt.show()